In [ ]:
!pip install pageindex
!pip install langchain langchain-google-genai google-generativeai -q

In [ ]:
import os
import json
import requests  #upload docs,call llm api, query retrieval service, fetch results
import asyncio #concurrent operations to run
from pageindex import PageIndexClient #client to interact to and fro with pageindex server,user queries
import pageindex.utils as utils #printing doc tree


In [ ]:
from google.colab import userdata
pi_key = userdata.get("pi_key")
print("API key loaded:", "SUCCESS" if pi_key else "NOT FOUND")

API key loaded: SUCCESS


In [ ]:
pi_client=PageIndexClient(api_key=pi_key)

:

In [ ]:
pdf_url="https://arxiv.org/pdf/2605.06285.pdf"
pdf_path=os.path.join("../data",pdf_url.split('/')[-1])
os.makedirs(os.path.dirname(pdf_path),exist_ok=True)

response=requests.get(pdf_url)# send http req and downloads file
with open(pdf_path,"wb")as f:   #write binary mode
  f.write(response.content) #saving file to local system

In [ ]:
print(pdf_path)

../data/2605.06285.pdf


In [ ]:
#Upload file to PageIndex for hierarchial indexing

doc_info = pi_client.submit_document(pdf_path)
doc_id=doc_info["doc_id"]
print('Document Submitted: ',doc_id)

Document Submitted:  pi-cmqhrhote031e01pkyazna59h


In [ ]:
import time
print(f"Waiting for document {doc_id} to be indexed.")
max_retries=10 # each check takes 5 seconds
retry_count=0
while not pi_client.is_retrieval_ready(doc_id): # loop continues polling if not ready
  if retry_count>=max_retries:
    print("Timeout: Document processing took too long.")
    break

  print(f"Still processing..(Attempt{retry_count+1}/{max_retries})")
  time.sleep(5)
  retry_count+=1

if pi_client.is_retrieval_ready(doc_id):
  print("Success! Document is ready.")
  tree=pi_client.get_tree(doc_id,node_summary=True)['result'] #gets tree structure
  utils.print_tree(tree) #print tree index
else:
    tree=None

Waiting for document pi-cmqhrhote031e01pkyazna59h to be indexed.
Still processing..(Attempt1/10)
Still processing..(Attempt2/10)
Still processing..(Attempt3/10)
Still processing..(Attempt4/10)
Still processing..(Attempt5/10)
Still processing..(Attempt6/10)
Success! Document is ready.
[{'title': 'LatentRAG: Latent Reasoning and Retrieva...',
  'node_id': '0000',
  'prefix_summary': '# LatentRAG: Latent Reasoning and Retrie...',
  'nodes': [{'title': 'Abstract',
             'node_id': '0001',
             'summary': 'The text introduces LatentRAG, a framewo...'},
            {'title': '1 Introduction',
             'node_id': '0002',
             'summary': 'The text introduces LatentRAG, an effici...'},
            {'title': '2 Related Work',
             'node_id': '0003',
             'summary': 'This text reviews the development of age...'},
            {'title': '3 Preliminaries',
             'node_id': '0004',
             'summary': 'This text outlines the preliminaries of ...'}

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI #bridge between langchain and gemini
import os
from google.colab import userdata

gemini_api_key = userdata.get('gemini_key')
os.environ["google_key"] = gemini_api_key # Correct environment variable and value
llm=ChatGoogleGenerativeAI(
  model="gemini-2.5-flash",
  temperature=0.3 # randomness
  )

In [ ]:
def retrieval_from_pi(query,doc_id,top_k=10):
  response=pi_client.submit_query( #sends query and doc id to pi via client
      doc_id=doc_id,
      query=query #response is a python dictionary with ret_id and status
  )
  retrieval_id=response.get("retrieval_id") #HTTP GET
  if not retrieval_id:
    return []
  while True: # checking status
    retrieval=pi_client.get_retrieval(retrieval_id) # asking task status
    status=retrieval.get("status")

    if status=="completed":
      break
    elif status=='failed':
      return []

    time.sleep(1)
  nodes=retrieval.get("retrieved_nodes",[])
  contexts=[]

  for node in nodes[:top_k]:
    relevant_contents=node.get("relevant_contents",[])

    for group in relevant_contents:
      for item in group:
        content=item.get("relevant_content")
        if content:
          contexts.append(content)
        return contexts

In [ ]:
print(response)

<Response [200]>


In [ ]:
def vectorless_rag(query,doc_id):
  contexts=retrieval_from_pi(query,doc_id)
  if not contexts:
    return "No relevant context found"
  combined_context="\n\n".join(contexts)

  prompt = f"""
  You are an expert researcher.

  Answer only from the provided context.

  Provide:
  1. A clear explanation.
  2. Important points.
  3. Examples if available.
  4. Use headings and bullet points.
  5. Give a detailed answer of at least 200 words when sufficient context exists.

  Context:
  {combined_context}

  Question:
  {query}
  """
  response=llm.invoke(prompt)
  return response.content


In [ ]:
query=input("Enter your question: ")
answer=vectorless_rag(query,doc_id)
print(answer)

Enter your question: What is the conclusion?
Based on the provided context, the conclusion regarding LatentRAG is as follows:

### 1. Clear Explanation

The paper introduces LatentRAG, an innovative and efficient agentic RAG (Retrieval Augmented Generation) framework. Its fundamental innovation lies in its operational paradigm, where both the reasoning and retrieval processes are conducted within a continuous latent space, rather than the conventional discrete language space. This architectural shift is central to its proposed advantages.

Experimental results demonstrate that LatentRAG successfully achieves performance levels that are comparable to those of existing agentic RAG methods. A key finding is its remarkable efficiency, as the framework has been shown to reduce operational latency by approximately 90%. This significant reduction in processing time makes LatentRAG a highly efficient alternative.

To address potential concerns regarding the interpretability of operations withi

In [ ]:
pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 45.3 MB/s eta 0:00:00


In [ ]:
import streamlit as st
import tempfile
import time

st.title("Vectorless RAG using PageIndex")

uploaded_files = st.file_uploader(
    "Upload PDFs",
    type="pdf",
    accept_multiple_files=True
)

doc_ids = []

if uploaded_files:

    if st.button("Index Documents"):

        log_box = st.empty()
        logs = []

        for file in uploaded_files:

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=".pdf"
            ) as tmp:

                tmp.write(file.getbuffer())
                pdf_path = tmp.name

            logs.append(f"Uploading {file.name}")
            log_box.code("\n".join(logs))

            doc_info = pi_client.submit_document(pdf_path)

            doc_id = doc_info["doc_id"]
            doc_ids.append(doc_id)

            logs.append(f"Document ID: {doc_id}")
            log_box.code("\n".join(logs))

            while not pi_client.is_retrieval_ready(doc_id):
                time.sleep(2)

            logs.append(f"{file.name} indexed")
            log_box.code("\n".join(logs))

        st.session_state["doc_ids"] = doc_ids

        st.success("All PDFs Indexed")

query = st.text_input("Ask a question")

if st.button("Submit Query"):

    doc_ids = st.session_state.get("doc_ids", [])

    all_contexts = []

    for doc_id in doc_ids:

        contexts = retrieve_from_pageindex(
            query,
            doc_id,
            top_k=5
        )

        all_contexts.extend(contexts)

    combined_context = "\n".join(all_contexts)

    answer = generate_answer(
        query,
        combined_context
    )

    st.subheader("Answer")
    st.write(answer)

    st.expander("Retrieved Context")

2026-06-17 09:41:29.667 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 09:41:29.669 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 09:41:29.670 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 09:41:29.671 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 09:41:29.673 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 09:41:29.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 09:41:29.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-17 09:41:29.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar